# Bayesian Flow Networks – GPT-2 Scale (Discrete Text)

> **Paper:** *Bayesian Flow Networks* – Graves et al., 2023 ([arXiv:2308.07037](https://arxiv.org/abs/2308.07037))  
> **Repo base:** [Algomancer/Bayesian-Flow-Networks](https://github.com/Algomancer/Bayesian-Flow-Networks)  
> **This notebook:** Implements the missing "Bayesian Flow GPT-2 Scale" feature — a causal GPT-2-sized transformer trained with the BFN discrete continuous-time loss on WikiText-2.

---

## 🗺️ Notebook Structure
| Section | Nội dung |
|---------|----------|
| 1 | Lý thuyết BFN – review toán học từ paper |
| 2 | GPT-2 Transformer backbone (causal) |
| 3 | BayesianFlowNetwork class hoàn chỉnh |
| 4 | Dataset – WikiText-2 (character-level demo) |
| 5 | Training loop |
| 6 | Sampling & text generation |
| 7 | Loss curve visualization |


---
## 1. Lý Thuyết Bayesian Flow Networks

### 1.1 Vấn đề tổng quát

BFN là một mô hình sinh mới, thay thế cho diffusion models. Thay vì corrupt data trong không gian input, BFN tích lũy thông tin **trong không gian tham số** của một Bayesian prior.

### 1.2 Bayesian Update cho dữ liệu rời rạc (Discrete Data)

Với alphabet $\mathcal{K} = \{1, \ldots, K\}$, prior là phân phối Dirichlet đồng đều:

$$p(\boldsymbol{\theta}) = \text{Dir}(\boldsymbol{\theta}; \mathbf{1}_K)$$

**Sender distribution** (Eq. 73 trong paper): tại accuracy $\alpha$, gửi sample $y$ từ:

$$p_S(y \mid x; \alpha) = \mathcal{N}\!\left(y;\, \alpha\bigl(K\,\mathbf{e}_x - \mathbf{1}\bigr),\, \alpha K \,\mathbf{I}\right)$$

trong đó $\mathbf{e}_x \in \{0,1\}^K$ là one-hot vector của token $x$.

### 1.3 Bayesian Update (Eq. 74)

Sau khi nhận $y$, tham số $\theta$ được update:

$$\boldsymbol{\theta}' = \text{softmax}(y + \log \boldsymbol{\theta})$$

### 1.4 Bayesian Flow Distribution (Eq. 75)

Flow tham số theo thời gian $t \in [0, 1]$ với $\beta(t) = \beta t^2$ (schedule bậc hai):

$$p_F(\boldsymbol{\theta} \mid x; t) = \mathcal{N}\!\left(e_x;\, \beta(t)\bigl(K\,\mathbf{e}_x - \mathbf{1}\bigr),\, \beta(t) K\,\mathbf{I}\right) \cdot \text{softmax}(\cdot)$$

Trong implementation, ta sample $\boldsymbol{\theta}$ bằng:
$$y \sim \mathcal{N}\!\left(\beta(t)(K\,\mathbf{e}_x - 1),\; \beta(t) K\right), \quad \boldsymbol{\theta} = \text{softmax}(y)$$

### 1.5 Continuous-Time Loss $L^\infty$ (Eq. 83)

Loss function tối ưu cho discrete BFN với continuous time:

$$\boxed{L^\infty(\mathbf{x}) = K \beta \sum_{d=1}^{D} \int_0^1 t \, \mathbb{E}_{\boldsymbol{\theta} \sim p_F(\cdot \mid x_d; t)} \left\| \mathbf{e}_{x_d} - \hat{\mathbf{e}}(\boldsymbol{\theta}, t) \right\|^2 \, dt}$$

Dạng Monte Carlo (sample $t \sim \mathcal{U}(0,1)$):

$$\hat{L}^\infty = K \beta t \left\| \mathbf{e}_x - \hat{\mathbf{e}}(\boldsymbol{\theta}, t) \right\|^2$$

trong đó $\hat{\mathbf{e}}(\boldsymbol{\theta}, t) = \text{softmax}(f_\theta(\boldsymbol{\theta}, t))$ là output của neural network.

### 1.6 Sampling (Algorithm 2 – Discrete)

Với $n$ bước lấy mẫu, accuracy schedule: $\alpha_i = \beta \frac{2i - 1}{n^2}$

1. Khởi đầu: $\boldsymbol{\theta}^{(0)} = \mathbf{1}/K$ (uniform)
2. Với mỗi bước $i = 1, \ldots, n$:
   - $t_i = (i-1)/n$
   - $\hat{\mathbf{e}} = f_\theta(\boldsymbol{\theta}^{(i-1)}, t_i)$  (network prediction)
   - $k \sim \text{Cat}(\hat{\mathbf{e}})$
   - $y \sim \mathcal{N}(\alpha_i(K\,\mathbf{e}_k - 1),\; \alpha_i K)$
   - $\boldsymbol{\theta}^{(i)} = \text{softmax}(y + \log \boldsymbol{\theta}^{(i-1)})$
3. Final: $k^* \sim \text{Cat}(f_\theta(\boldsymbol{\theta}^{(n)}, 1))$

### 1.7 GPT-2 Scale vs. Original Repo

| Thành phần | Original (LLAMA2-style) | Notebook này (GPT-2) |
|---|---|---|
| Architecture | Non-causal Transformer | Causal GPT-2 |
| Positional encoding | RoPE | Learned absolute |
| Normalization | RMSNorm | LayerNorm |
| Vocab | 32000 (llama tok) | 50257 (GPT-2 BPE) |
| Data | TinyStories | WikiText-2 |
| Scale | 15M params | ~117M params (GPT-2 small) |


---
## 2. Cài đặt & Import

In [ ]:
# Cài đặt các thư viện cần thiết
# Chạy cell này một lần, sau đó có thể comment lại
import subprocess, sys

def pip_install(*pkgs):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *pkgs])

pip_install("torch", "transformers", "datasets", "tiktoken", "matplotlib", "tqdm")
print("✅ All packages installed!")


In [ ]:
import math
import time
import json
from dataclasses import dataclass
from typing import Optional

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset

import numpy as np
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

# Kiểm tra device
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"🖥️  Using device: {DEVICE}")


---
## 3. GPT-2 Transformer Backbone (Causal)

Chúng ta implement GPT-2 architecture làm backbone cho BFN.  
Điểm khác biệt chính so với LLAMA2 trong repo gốc:
- **Causal attention mask** (autoregressive theo position)
- **Learned positional embeddings** thay vì RoPE
- **LayerNorm** thay vì RMSNorm
- **GELU activation** thay vì SwiGLU

> ⚠️ **Quan trọng:** BFN *không* autoregressively generate từng token.  
> Thay vào đó model nhận **toàn bộ chuỗi** $\boldsymbol{\theta}^{(1:D)}$ và dự đoán **song song** tất cả $D$ token.  
> Causal mask ở đây dùng để mô hình hoá dependencies giữa các vị trí, không phải để generate left-to-right.


In [ ]:
# ─────────────────────────────────────────────────────────────
# GPT-2 Config
# ─────────────────────────────────────────────────────────────

@dataclass
class GPT2Config:
    """
    Hyperparameters cho GPT-2 scale BFN.
    Mặc định = GPT-2 small (~117M params).
    """
    vocab_size: int = 50257       # GPT-2 BPE vocabulary
    max_seq_len: int = 256        # Độ dài sequence (giảm để demo nhanh)
    n_embd: int = 768             # Embedding dimension
    n_heads: int = 12             # Số attention heads
    n_layers: int = 12            # Số transformer layers
    dropout: float = 0.1          # Dropout rate
    bias: bool = True             # Có dùng bias trong Linear/LayerNorm

    # BFN-specific: số class K = vocab_size
    # D = max_seq_len (số vị trí)
    # Input cho model: (B, D, K) → reshaped → (B, D, n_embd)


In [ ]:
# ─────────────────────────────────────────────────────────────
# Causal Self-Attention
# ─────────────────────────────────────────────────────────────

class CausalSelfAttention(nn.Module):
    """
    Multi-head causal self-attention.
    Input: (B, T, C) where T = sequence length, C = n_embd
    Output: (B, T, C)
    """

    def __init__(self, config: GPT2Config):
        super().__init__()
        assert config.n_embd % config.n_heads == 0

        self.n_heads = config.n_heads
        self.n_embd = config.n_embd
        self.head_dim = config.n_embd // config.n_heads
        self.dropout_p = config.dropout

        # Q, K, V projection – gộp thành 1 linear để hiệu quả
        self.c_attn = nn.Linear(config.n_embd, 3 * config.n_embd, bias=config.bias)
        # Output projection
        self.c_proj = nn.Linear(config.n_embd, config.n_embd, bias=config.bias)

        self.attn_dropout = nn.Dropout(config.dropout)
        self.resid_dropout = nn.Dropout(config.dropout)

        # Causal mask: lower-triangular matrix
        # Đăng ký như buffer (không phải parameter, nhưng được lưu trong state_dict)
        self.register_buffer(
            "bias",
            torch.tril(torch.ones(config.max_seq_len, config.max_seq_len))
            .view(1, 1, config.max_seq_len, config.max_seq_len)
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        B, T, C = x.shape  # Batch, Sequence length, Channels

        # Tính Q, K, V cùng lúc
        qkv = self.c_attn(x)                      # (B, T, 3C)
        q, k, v = qkv.split(self.n_embd, dim=2)   # mỗi cái (B, T, C)

        # Reshape sang multi-head format: (B, n_heads, T, head_dim)
        q = q.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        k = k.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        v = v.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)

        # Scaled dot-product attention
        # att(Q,K,V) = softmax(QK^T / sqrt(d_k)) V
        scale = 1.0 / math.sqrt(self.head_dim)
        att = (q @ k.transpose(-2, -1)) * scale   # (B, n_heads, T, T)

        # Apply causal mask: các token sau không attend được token trước
        att = att.masked_fill(self.bias[:, :, :T, :T] == 0, float('-inf'))
        att = F.softmax(att, dim=-1)
        att = self.attn_dropout(att)

        # Weighted aggregation of values
        out = att @ v                              # (B, n_heads, T, head_dim)
        out = out.transpose(1, 2).contiguous().view(B, T, C)  # (B, T, C)
        out = self.resid_dropout(self.c_proj(out))
        return out


In [ ]:
# ─────────────────────────────────────────────────────────────
# MLP block (Feed-Forward)
# ─────────────────────────────────────────────────────────────

class MLP(nn.Module):
    """
    Feed-forward block với GELU activation (giống GPT-2).
    FFN(x) = GELU(x W_1 + b_1) W_2 + b_2
    """

    def __init__(self, config: GPT2Config):
        super().__init__()
        # Expand 4x như GPT-2 gốc
        self.c_fc   = nn.Linear(config.n_embd, 4 * config.n_embd, bias=config.bias)
        self.gelu   = nn.GELU()
        self.c_proj = nn.Linear(4 * config.n_embd, config.n_embd, bias=config.bias)
        self.dropout = nn.Dropout(config.dropout)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.c_fc(x)
        x = self.gelu(x)
        x = self.c_proj(x)
        x = self.dropout(x)
        return x


# ─────────────────────────────────────────────────────────────
# Transformer Block
# ─────────────────────────────────────────────────────────────

class TransformerBlock(nn.Module):
    """
    Standard Pre-LN Transformer block:
      h = x + Attention(LayerNorm(x))
      out = h + MLP(LayerNorm(h))
    """

    def __init__(self, config: GPT2Config):
        super().__init__()
        self.ln_1 = nn.LayerNorm(config.n_embd, bias=config.bias)
        self.attn  = CausalSelfAttention(config)
        self.ln_2  = nn.LayerNorm(config.n_embd, bias=config.bias)
        self.mlp   = MLP(config)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # Pre-LN residual connections (ổn định hơn Post-LN)
        x = x + self.attn(self.ln_1(x))
        x = x + self.mlp(self.ln_2(x))
        return x


In [ ]:
# ─────────────────────────────────────────────────────────────
# GPT-2 Model (backbone cho BFN)
# ─────────────────────────────────────────────────────────────

class GPT2BFN(nn.Module):
    """
    GPT-2 style transformer được chỉnh sửa để làm backbone BFN.

    **Thay đổi so với GPT-2 gốc:**
    - Input: (B, D, K) thay vì (B, D) token indices
      Vì BFN nhận tham số θ ∈ [0,1]^K (phân phối), không phải discrete token
    - Không có embedding lookup table – dùng linear projection từ K → n_embd
    - Thêm time embedding t ∈ [0,1] vào mỗi position
    - Output: (B, D, K) logits cho phân phối output

    Kiến trúc tổng quát:
      θ (B,D,K) ──Linear──► (B,D,C) ─┐
      t (B,) ──MLP──► (B,C) ──────────┤ ──► Add ──► pos_emb ──► Transformer ──► head ──► (B,D,K)
    """

    def __init__(self, config: GPT2Config):
        super().__init__()
        self.config = config

        # Input projection: K → n_embd
        # Nhận θ ∈ [0,1]^K và project lên embedding space
        self.input_proj = nn.Linear(config.vocab_size, config.n_embd, bias=config.bias)

        # Learned positional embeddings
        self.pos_emb = nn.Embedding(config.max_seq_len, config.n_embd)

        # Time embedding: scalar t → n_embd
        # Dùng sinusoidal + learned MLP (như trong diffusion models)
        time_emb_dim = config.n_embd
        self.time_mlp = nn.Sequential(
            SinusoidalEmbedding(time_emb_dim),   # t → (B, n_embd)
            nn.Linear(time_emb_dim, time_emb_dim),
            nn.GELU(),
            nn.Linear(time_emb_dim, config.n_embd),
        )

        # Dropout trên input
        self.drop = nn.Dropout(config.dropout)

        # Stack of transformer blocks
        self.blocks = nn.ModuleList([
            TransformerBlock(config) for _ in range(config.n_layers)
        ])

        # Final LayerNorm
        self.ln_f = nn.LayerNorm(config.n_embd, bias=config.bias)

        # Output head: n_embd → K (logits cho mỗi class)
        self.head = nn.Linear(config.n_embd, config.vocab_size, bias=False)

        # Weight initialization
        self.apply(self._init_weights)

        # Đặc biệt: scale residual projections theo depth (GPT-2 trick)
        for pn, p in self.named_parameters():
            if pn.endswith('c_proj.weight'):
                torch.nn.init.normal_(p, mean=0.0, std=0.02 / math.sqrt(2 * config.n_layers))

        n_params = sum(p.numel() for p in self.parameters())
        print(f"📊 GPT-2 BFN backbone: {n_params/1e6:.1f}M parameters")

    def _init_weights(self, module):
        """Xavier/normal initialization như GPT-2."""
        if isinstance(module, nn.Linear):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, theta: torch.Tensor, t: torch.Tensor) -> torch.Tensor:
        """
        Args:
            theta: (B, D, K) – Bayesian flow parameters (sau softmax, ∈ [0,1]^K)
            t:     (B,)      – thời gian t ∈ [0,1]

        Returns:
            logits: (B, D, K) – unnormalized logits của output distribution
        """
        B, D, K = theta.shape
        assert D <= self.config.max_seq_len, f"Sequence length {D} > max {self.config.max_seq_len}"

        # 1. Project θ từ K dimensions → n_embd
        x = self.input_proj(theta)   # (B, D, n_embd)

        # 2. Positional embedding
        pos = torch.arange(D, device=theta.device)  # (D,)
        pos_emb = self.pos_emb(pos)                  # (D, n_embd)
        x = x + pos_emb.unsqueeze(0)                 # (B, D, n_embd)

        # 3. Time embedding: thêm time info vào mỗi position
        # t_emb: (B, n_embd) → broadcast sang mỗi position
        t_emb = self.time_mlp(t)         # (B, n_embd)
        x = x + t_emb.unsqueeze(1)       # (B, D, n_embd) – broadcast

        x = self.drop(x)

        # 4. Transformer blocks
        for block in self.blocks:
            x = block(x)

        # 5. Final norm + output head
        x = self.ln_f(x)                 # (B, D, n_embd)
        logits = self.head(x)            # (B, D, K)
        return logits


# ─────────────────────────────────────────────────────────────
# Sinusoidal Time Embedding (như trong DDPM)
# ─────────────────────────────────────────────────────────────

class SinusoidalEmbedding(nn.Module):
    """
    Chuyển scalar t ∈ [0,1] thành vector embedding bằng sinusoidal functions.
    Công thức: emb[2i] = sin(t * 10000^(-2i/d)), emb[2i+1] = cos(...)
    """

    def __init__(self, dim: int):
        super().__init__()
        self.dim = dim

    def forward(self, t: torch.Tensor) -> torch.Tensor:
        """
        Args:
            t: (B,) scalar time values ∈ [0,1]
        Returns:
            emb: (B, dim)
        """
        device = t.device
        half = self.dim // 2
        # Frequencies: 10000^(-2i/d) for i = 0..half-1
        freqs = torch.exp(
            -math.log(10000) * torch.arange(half, device=device) / (half - 1)
        )  # (half,)
        # Outer product: t * freqs
        args = t[:, None] * freqs[None, :]  # (B, half)
        emb = torch.cat([torch.sin(args), torch.cos(args)], dim=-1)  # (B, dim)
        return emb


---
## 4. BayesianFlowNetwork Class

Wrap GPT-2 backbone trong BFN framework, implement:
- **`process()`**: tính $L^\infty$ loss từ batch
- **`sample()`**: generate text bằng iterative Bayesian updates


In [ ]:
class BayesianFlowNetwork(nn.Module):
    """
    Bayesian Flow Network cho discrete text generation.

    Wrap một neural network backbone (GPT-2) với:
    - Bayesian flow parameter update
    - Continuous-time ELBO loss (L∞)
    - Iterative sampling

    Args:
        model:  Neural network backbone nhận (θ, t) → logits
        D:      Sequence length (number of data dimensions)
        K:      Vocabulary size (number of classes per dimension)
        beta:   Accuracy schedule parameter (paper dùng β = 1.0 cho text)
    """

    def __init__(self, model: nn.Module, D: int, K: int, beta: float = 1.0):
        super().__init__()
        self.model = model
        self.D = D       # Sequence length
        self.K = K       # Vocab size
        self.beta = beta  # β trong schedule β(t) = βt²

    # ──────────────────────────────────────────────────────────
    # Forward pass
    # ──────────────────────────────────────────────────────────

    def forward(self, theta: torch.Tensor, t: torch.Tensor) -> torch.Tensor:
        """
        Gọi backbone model.

        Args:
            theta: (B, D, K) – flow parameters (scaled to [-1,1] trước khi đưa vào model)
            t:     (B,)      – time

        Returns:
            output: (B, D, K) logits
        """
        # Scale θ từ [0,1] sang [-1,1] (normalization trick từ repo gốc)
        # Giúp model học ổn định hơn
        theta_scaled = (theta * 2) - 1   # (B, D, K)
        output = self.model(theta_scaled, t)
        return output

    # ──────────────────────────────────────────────────────────
    # Discrete Output Distribution (Algorithm 1, step 5)
    # ──────────────────────────────────────────────────────────

    def discrete_output_distribution(
        self, theta: torch.Tensor, t: torch.Tensor
    ) -> torch.Tensor:
        """
        Tính phân phối output p_O(x|θ, t).

        Với K=2: dùng sigmoid (binary classification)
        Với K>2: dùng softmax (multinomial)

        Returns:
            p0: (B, D, K) – predicted probabilities cho mỗi class
        """
        B, D, K = theta.shape
        logits = self.forward(theta, t)   # (B, D, K)

        if K == 2:
            # Dùng sigmoid cho binary (numerically stable hơn softmax)
            p1 = torch.sigmoid(logits[..., :1])
            p0 = torch.cat([p1, 1 - p1], dim=-1)
        else:
            # Softmax cho K > 2
            p0 = F.softmax(logits, dim=-1)

        return p0

    # ──────────────────────────────────────────────────────────
    # Training: Compute L∞ loss
    # ──────────────────────────────────────────────────────────

    def process(self, x: torch.Tensor) -> torch.Tensor:
        """
        Tính continuous-time loss L∞ theo Algorithm 3 (paper Section 6.8).

        L∞ = K·β·t · ‖e_x - ê(θ,t)‖²

        Args:
            x: (B, D) – token indices (long tensor)

        Returns:
            loss: scalar mean loss
        """
        B, D = x.shape

        # ── Step 1: Sample t ~ Uniform(0, 1) ──────────────────
        # Mỗi sample trong batch có một giá trị t khác nhau
        t = torch.rand((B,), device=x.device, dtype=torch.float32)

        # ── Step 2: Tính β(t) = β·t² ──────────────────────────
        # Schedule bậc hai để phân phối accuracy theo thời gian
        # (Eq. 76 trong paper)
        beta_t = self.beta * (t ** 2)   # (B,)

        # ── Step 3: Sample y ~ p_S(y|x; β(t)) ────────────────
        # Sender distribution: Gaussian around one-hot encoding
        # mean = β(t)·(K·e_x - 1), std = sqrt(β(t)·K)
        one_hot_x = F.one_hot(x, num_classes=self.K).float()  # (B, D, K)

        # mean: (B, D, K) – broadcast β_t over D and K dimensions
        mean = beta_t[:, None, None] * (self.K * one_hot_x - 1)

        # std: (B, 1, 1) – constant across positions
        std = (beta_t * self.K)[:, None, None].sqrt()

        # Reparameterization trick: y = mean + std * ε, ε ~ N(0, I)
        eps = torch.randn_like(mean)
        y = mean + std * eps            # (B, D, K)

        # ── Step 4: Bayesian update → θ ───────────────────────
        # θ = softmax(y)
        # (Khi prior là uniform Dir(1), posterior = softmax(y))
        theta = F.softmax(y, dim=-1)    # (B, D, K)

        # ── Step 5: Predict output distribution ───────────────
        # ê(θ, t) = model prediction của e_x given θ, t
        p_0 = self.discrete_output_distribution(theta, t)  # (B, D, K)

        # ── Step 6: Tính L∞ loss ──────────────────────────────
        # L∞ = K·β·t · ‖e_x - p̂_0‖²
        # Đây là MSE giữa one-hot target và predicted distribution
        # nhân với weight K·β·t
        e_x = one_hot_x                  # (B, D, K) – ground truth one-hot
        e_hat = p_0                      # (B, D, K) – predicted distribution

        # Weight factor (Eq. 83): K·β·t (broadcast)
        weight = self.K * self.beta * t[:, None, None]  # (B, 1, 1) → broadcast

        L_infinity = weight * ((e_x - e_hat) ** 2)     # (B, D, K)

        return L_infinity.mean()   # scalar

    # ──────────────────────────────────────────────────────────
    # Sampling: Generate text (Algorithm 2)
    # ──────────────────────────────────────────────────────────

    @torch.inference_mode()
    def sample(
        self,
        batch_size: int = 4,
        nb_steps: int = 100,
        device: str = "cpu",
        temperature: float = 1.0,
        eps_: float = 1e-12
    ) -> torch.Tensor:
        """
        Generate sequences bằng iterative Bayesian updates.
        Implements Algorithm 2 (Discrete BFN Sampling).

        Args:
            batch_size: Số sequences cần generate
            nb_steps:   Số bước lặp (nhiều bước → tốt hơn, nhưng chậm hơn)
            device:     'cpu' hoặc 'cuda'
            temperature: Temperature cho sampling (1.0 = default)
            eps_:       Epsilon tránh log(0)

        Returns:
            k_final: (batch_size, D) – generated token indices
        """
        self.eval()

        # ── Khởi đầu: θ = uniform distribution ─────────────────
        # Prior p(θ) = Dir(1) → mean = 1/K
        theta = torch.ones(
            (batch_size, self.D, self.K), device=device
        ) / self.K                           # (B, D, K)

        for i in range(1, nb_steps + 1):
            # ── Time step ────────────────────────────────────────
            t_val = (i - 1) / nb_steps
            t = torch.full(
                (batch_size,), t_val, device=device, dtype=torch.float32
            )

            # ── Predict output distribution ───────────────────────
            # k_probs: (B, D, K) – probability cho mỗi class
            k_probs = self.discrete_output_distribution(theta, t)

            # ── Sample k ~ Cat(k_probs) ───────────────────────────
            # Apply temperature scaling
            if temperature != 1.0:
                k_probs = k_probs.pow(1.0 / temperature)
                k_probs = k_probs / k_probs.sum(dim=-1, keepdim=True)

            k = torch.distributions.Categorical(probs=k_probs).sample()  # (B, D)

            # ── Tính accuracy cho bước này ─────────────────────────
            # α_i = β(2i-1)/n² (Eq. 82 trong paper)
            alpha = self.beta * (2 * i - 1) / (nb_steps ** 2)

            # ── Sample từ sender distribution ─────────────────────
            e_k = F.one_hot(k, num_classes=self.K).float()  # (B, D, K)
            mean = alpha * (self.K * e_k - 1)
            std_val = math.sqrt(alpha * self.K)
            std = torch.full_like(mean, fill_value=std_val)
            eps = torch.randn_like(e_k)
            y = mean + std * eps    # (B, D, K)

            # ── Bayesian update: θ' = softmax(y + log θ) ──────────
            # (Eq. 74 trong paper)
            theta = F.softmax(y + torch.log(theta + eps_), dim=-1)

        # ── Final prediction tại t=1 ───────────────────────────────
        t_final = torch.ones((batch_size,), device=device)
        k_probs_final = self.discrete_output_distribution(theta, t_final)
        k_final = torch.distributions.Categorical(probs=k_probs_final).sample()  # (B, D)

        return k_final


---
## 5. Dataset – WikiText-2 (Character-level Demo)

Để notebook có thể chạy ngay mà không cần download lớn, chúng ta dùng **character-level** encoding trên WikiText-2 (nhỏ, dễ tải).

Khi scale lên GPT-2 full (50k vocab), thay `CharDataset` bằng tokenizer GPT-2 và WikiText-103.


In [ ]:
from datasets import load_dataset

# ─────────────────────────────────────────────────────────────
# Load WikiText-2
# ─────────────────────────────────────────────────────────────

print("📥 Loading WikiText-2...")
wikitext = load_dataset("wikitext", "wikitext-2-raw-v1", trust_remote_code=True)

# Ghép tất cả text lại
train_text = "\n".join(wikitext["train"]["text"])
val_text   = "\n".join(wikitext["validation"]["text"])

# Xây vocab ký tự
chars = sorted(set(train_text))
VOCAB_SIZE = len(chars)
print(f"📚 Character vocab size: {VOCAB_SIZE}")
print(f"📊 Train text length: {len(train_text):,} chars")

# Encode/Decode functions
stoi = {c: i for i, c in enumerate(chars)}
itos = {i: c for c, i in stoi.items()}
encode = lambda s: [stoi[c] for c in s if c in stoi]
decode = lambda l: "".join([itos[i] for i in l])

print(f"\nVí dụ encode: '{train_text[:20]}' → {encode(train_text[:20])[:10]}...")


In [ ]:
# ─────────────────────────────────────────────────────────────
# Character-level Dataset
# ─────────────────────────────────────────────────────────────

class CharDataset(Dataset):
    """
    Dataset cho character-level language modeling.
    Mỗi item là một sequence có độ dài seq_len.
    """

    def __init__(self, text: str, seq_len: int, encode_fn):
        self.seq_len = seq_len
        # Tokenize toàn bộ text
        data = encode_fn(text)
        self.data = torch.tensor(data, dtype=torch.long)
        print(f"  Dataset: {len(self.data):,} tokens, {len(self):,} sequences of length {seq_len}")

    def __len__(self):
        return len(self.data) - self.seq_len

    def __getitem__(self, idx: int) -> torch.Tensor:
        # Lấy slice độ dài seq_len
        return self.data[idx : idx + self.seq_len]

# Config nhỏ cho demo (có thể scale up)
SEQ_LEN   = 64    # Độ dài sequence (giảm để demo nhanh; GPT-2 dùng 1024)
BATCH_SIZE = 32

print("📦 Creating datasets...")
train_dataset = CharDataset(train_text, SEQ_LEN, encode)
val_dataset   = CharDataset(val_text,   SEQ_LEN, encode)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    drop_last=True,
    pin_memory=(DEVICE == "cuda"),
    num_workers=0,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    drop_last=True,
    pin_memory=(DEVICE == "cuda"),
    num_workers=0,
)

print(f"\n✅ Train batches: {len(train_loader)}, Val batches: {len(val_loader)}")


---
## 6. Khởi Tạo Model

Chúng ta dùng **config nhỏ** để demo có thể chạy trên CPU trong vài phút.  
Để scale lên GPT-2 scale thật (~117M), dùng `GPT2Config()` mặc định và train trên GPU.


In [ ]:
# ─────────────────────────────────────────────────────────────
# Model Config (Demo – nhỏ để chạy nhanh)
# ─────────────────────────────────────────────────────────────

# Demo config (chạy được trên CPU ~5-10 phút)
config = GPT2Config(
    vocab_size  = VOCAB_SIZE,   # character vocab (≈100)
    max_seq_len = SEQ_LEN,      # 64 tokens
    n_embd      = 128,          # embedding dim (GPT-2 small: 768)
    n_heads     = 4,            # attention heads (GPT-2 small: 12)
    n_layers    = 4,            # transformer layers (GPT-2 small: 12)
    dropout     = 0.1,
    bias        = True,
)

# ─────────────────────────────────────────────────────────────
# Để dùng GPT-2 scale thật (~117M params), uncomment:
# ─────────────────────────────────────────────────────────────
# config = GPT2Config(
#     vocab_size  = 50257,   # GPT-2 BPE vocab
#     max_seq_len = 256,
#     n_embd      = 768,
#     n_heads     = 12,
#     n_layers    = 12,
#     dropout     = 0.1,
# )

# ─────────────────────────────────────────────────────────────
# Khởi tạo backbone và BFN
# ─────────────────────────────────────────────────────────────

print("🔧 Building GPT-2 backbone...")
backbone = GPT2BFN(config).to(DEVICE)

print("\n🔧 Wrapping in BayesianFlowNetwork...")
bfn = BayesianFlowNetwork(
    model = backbone,
    D     = config.max_seq_len,   # sequence length = D
    K     = config.vocab_size,    # vocab size = K
    beta  = 1.0,                   # β = 1.0 (giá trị paper recommend cho discrete)
).to(DEVICE)

# In tổng số parameters
total_params = sum(p.numel() for p in bfn.parameters() if p.requires_grad)
print(f"\n🎯 Total trainable parameters: {total_params:,} ({total_params/1e6:.2f}M)")
print(f"   Running on: {DEVICE}")


---
## 7. Training Loop

Training loop implement:
- **AdamW optimizer** với weight decay (như GPT-2 training setup)
- **Gradient clipping** (norm ≤ 1.0) để ổn định
- **Learning rate warmup** tuyến tính
- Logging loss mỗi N bước

> **Thời gian ước tính (demo config):** ~5-10 phút trên CPU, <2 phút trên GPU


In [ ]:
# ─────────────────────────────────────────────────────────────
# Training configuration
# ─────────────────────────────────────────────────────────────

LEARNING_RATE  = 3e-4      # Peak LR (như trong Karpathy's nanoGPT)
WEIGHT_DECAY   = 0.01      # Weight decay cho AdamW
BETA1, BETA2   = 0.9, 0.95 # AdamW betas
GRAD_CLIP      = 1.0       # Gradient clipping threshold
WARMUP_STEPS   = 100       # LR warmup steps
MAX_STEPS      = 1000      # Số bước train tối đa (tăng lên để train lâu hơn)
LOG_INTERVAL   = 50        # Log loss mỗi N steps
EVAL_INTERVAL  = 200       # Eval mỗi N steps

# ─────────────────────────────────────────────────────────────
# Optimizer với weight decay separation
# (Biases và LayerNorm params không weight decay)
# ─────────────────────────────────────────────────────────────

def configure_optimizer(model, lr, weight_decay, betas):
    """
    Tách parameters thành 2 nhóm:
    - decay: weight matrices (dim >= 2)
    - no_decay: biases, LayerNorm weights (dim < 2)
    """
    param_dict = {pn: p for pn, p in model.named_parameters() if p.requires_grad}

    decay_params   = [p for n, p in param_dict.items() if p.dim() >= 2]
    nodecay_params = [p for n, p in param_dict.items() if p.dim() < 2]

    optim_groups = [
        {"params": decay_params,   "weight_decay": weight_decay},
        {"params": nodecay_params, "weight_decay": 0.0},
    ]

    n_decay   = sum(p.numel() for p in decay_params)
    n_nodecay = sum(p.numel() for p in nodecay_params)
    print(f"   Decay params: {n_decay:,}   |   No-decay params: {n_nodecay:,}")

    # Dùng fused AdamW nếu có CUDA (nhanh hơn ~2x)
    use_fused = (DEVICE == "cuda") and hasattr(torch.optim, 'AdamW')
    optimizer = torch.optim.AdamW(optim_groups, lr=lr, betas=betas,
                                   fused=use_fused if DEVICE == "cuda" else False)
    return optimizer


print("⚙️  Setting up optimizer...")
optimizer = configure_optimizer(bfn, LEARNING_RATE, WEIGHT_DECAY, (BETA1, BETA2))


# ─────────────────────────────────────────────────────────────
# LR Scheduler: linear warmup → constant
# ─────────────────────────────────────────────────────────────

def get_lr(step: int) -> float:
    """Linear warmup schedule."""
    if step < WARMUP_STEPS:
        return LEARNING_RATE * step / WARMUP_STEPS
    return LEARNING_RATE

print("✅ Optimizer ready!")


In [ ]:
# ─────────────────────────────────────────────────────────────
# Training Loop
# ─────────────────────────────────────────────────────────────

train_losses = []
val_losses   = []
log_steps    = []

def evaluate(model, loader, max_batches=50):
    """Tính validation loss trên một subset của val set."""
    model.eval()
    total_loss = 0.0
    count = 0
    with torch.no_grad():
        for i, x in enumerate(loader):
            if i >= max_batches:
                break
            x = x.to(DEVICE)
            loss = model.process(x)
            total_loss += loss.item()
            count += 1
    model.train()
    return total_loss / max(count, 1)

print(f"🚀 Starting training for {MAX_STEPS} steps...")
print(f"   Logging every {LOG_INTERVAL} steps, evaluating every {EVAL_INTERVAL} steps\n")

bfn.train()
step = 0
data_iter = iter(train_loader)
t0 = time.time()

pbar = tqdm(total=MAX_STEPS, desc="Training")

while step < MAX_STEPS:
    # ── Data loading ────────────────────────────────────────
    try:
        x = next(data_iter)
    except StopIteration:
        data_iter = iter(train_loader)
        x = next(data_iter)

    x = x.to(DEVICE)

    # ── LR update ───────────────────────────────────────────
    lr = get_lr(step)
    for param_group in optimizer.param_groups:
        param_group["lr"] = lr

    # ── Forward + Backward ──────────────────────────────────
    optimizer.zero_grad(set_to_none=True)
    loss = bfn.process(x)
    loss.backward()

    # ── Gradient clipping ────────────────────────────────────
    # Clips gradient norm to prevent exploding gradients
    grad_norm = torch.nn.utils.clip_grad_norm_(bfn.parameters(), GRAD_CLIP)

    optimizer.step()

    # ── Logging ─────────────────────────────────────────────
    if step % LOG_INTERVAL == 0:
        elapsed = time.time() - t0
        loss_val = loss.item()
        train_losses.append(loss_val)
        log_steps.append(step)
        pbar.set_postfix({
            "loss": f"{loss_val:.4f}",
            "lr": f"{lr:.2e}",
            "grad": f"{grad_norm:.2f}",
            "t": f"{elapsed:.0f}s"
        })

    # ── Evaluation ──────────────────────────────────────────
    if step % EVAL_INTERVAL == 0 and step > 0:
        val_loss = evaluate(bfn, val_loader)
        val_losses.append((step, val_loss))
        tqdm.write(f"  📊 Step {step} | Val Loss: {val_loss:.4f}")

    pbar.update(1)
    step += 1

pbar.close()
total_time = time.time() - t0
print(f"\n✅ Training complete in {total_time:.1f}s ({total_time/60:.1f} min)")
print(f"   Final train loss: {train_losses[-1]:.4f}")


---
## 8. Visualize Training Curves

Plot loss curve để theo dõi convergence.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ── Training Loss ───────────────────────────────────────────
ax1 = axes[0]
ax1.plot(log_steps, train_losses, linewidth=1.5, color='steelblue', alpha=0.8, label='Train Loss')
# Smoothed curve
if len(train_losses) > 10:
    window = max(5, len(train_losses) // 10)
    smoothed = np.convolve(train_losses, np.ones(window)/window, mode='valid')
    smooth_steps = log_steps[window-1:]
    ax1.plot(smooth_steps, smoothed, linewidth=2.5, color='steelblue', label=f'Smoothed (w={window})')
ax1.set_xlabel("Training Steps", fontsize=12)
ax1.set_ylabel(r"$L^\infty$ Loss", fontsize=12)
ax1.set_title("BFN Training Loss", fontsize=14, fontweight='bold')
ax1.legend()
ax1.grid(True, alpha=0.3)

# ── Validation Loss ─────────────────────────────────────────
ax2 = axes[1]
if val_losses:
    val_x, val_y = zip(*val_losses)
    ax2.plot(val_x, val_y, 'o-', linewidth=2, color='coral', markersize=6, label='Val Loss')
    ax2.set_xlabel("Training Steps", fontsize=12)
    ax2.set_ylabel(r"$L^\infty$ Loss (Val)", fontsize=12)
    ax2.set_title("BFN Validation Loss", fontsize=14, fontweight='bold')
    ax2.legend()
    ax2.grid(True, alpha=0.3)
else:
    ax2.text(0.5, 0.5, "No validation data\n(increase MAX_STEPS)", 
             ha='center', va='center', transform=ax2.transAxes, fontsize=14)
    ax2.set_title("Validation Loss", fontsize=14)

plt.suptitle("Bayesian Flow Network – GPT-2 Scale Training", fontsize=15, y=1.02)
plt.tight_layout()
plt.savefig("bfn_gpt2_training.png", dpi=150, bbox_inches='tight')
plt.show()
print("📈 Plot saved to bfn_gpt2_training.png")


---
## 9. Sampling – Text Generation

Generate text bằng iterative Bayesian updates.

> **Lưu ý:** Sau vài nghìn steps training, model sẽ học được cấu trúc cơ bản.  
> Với đầy đủ training (100k+ steps, GPT-2 scale), chất lượng text sẽ tốt hơn nhiều.

### Giải thích quá trình sampling:

$$\boldsymbol{\theta}^{(0)} = \frac{1}{K}\mathbf{1} \xrightarrow{\text{step 1}} \boldsymbol{\theta}^{(1)} \xrightarrow{\text{step 2}} \cdots \xrightarrow{\text{step n}} \boldsymbol{\theta}^{(n)} \xrightarrow{\text{argmax}} \hat{\mathbf{x}}$$

Mỗi bước, model "tinh chỉnh" dự đoán của nó dựa trên thông tin Bayesian tích lũy.


In [ ]:
# ─────────────────────────────────────────────────────────────
# Text Generation
# ─────────────────────────────────────────────────────────────

def generate_text(
    bfn_model,
    decode_fn,
    batch_size: int = 3,
    nb_steps: int = 50,
    temperature: float = 1.0,
    device: str = "cpu"
) -> list:
    """
    Generate text sequences bằng BFN sampling.

    Args:
        bfn_model:   Trained BayesianFlowNetwork
        decode_fn:   Function map token ids → string
        batch_size:  Số sequences cần generate
        nb_steps:    Số Bayesian update steps
        temperature: Sampling temperature (< 1 = greedy-ish, > 1 = diverse)
        device:      'cpu' hoặc 'cuda'

    Returns:
        List of decoded strings
    """
    print(f"🎲 Sampling {batch_size} sequences ({nb_steps} steps, temp={temperature})...")

    with torch.no_grad():
        token_ids = bfn_model.sample(
            batch_size  = batch_size,
            nb_steps    = nb_steps,
            device      = device,
            temperature = temperature,
        )   # (B, D)

    # Decode token ids → text
    results = []
    for i in range(batch_size):
        ids = token_ids[i].tolist()
        text = decode_fn(ids)
        results.append(text)

    return results


# ── Generate với model hiện tại ─────────────────────────────
print("=" * 60)
print("📝 Generated Texts (after short training):")
print("=" * 60)

generated = generate_text(
    bfn_model   = bfn,
    decode_fn   = decode,
    batch_size  = 3,
    nb_steps    = 50,     # Tăng lên 200+ cho sample tốt hơn
    temperature = 0.8,
    device      = DEVICE,
)

for i, text in enumerate(generated):
    print(f"\n[Sample {i+1}]")
    print("-" * 40)
    print(text)
    print()

print("=" * 60)
print("💡 Tip: Tăng nb_steps và train lâu hơn để có text tốt hơn")


In [ ]:
# ─────────────────────────────────────────────────────────────
# Visualize: θ evolution during sampling
# Cho thấy cách BFN "refine" dự đoán qua từng bước
# ─────────────────────────────────────────────────────────────

def visualize_theta_evolution(bfn_model, nb_steps=20, device="cpu"):
    """
    Trace evolution của θ tại position 0 trong quá trình sampling.
    Giúp hiểu BFN học như thế nào.
    """
    bfn_model.eval()

    theta = torch.ones((1, bfn_model.D, bfn_model.K), device=device) / bfn_model.K
    eps_ = 1e-12
    theta_history = []   # lưu top-5 probs qua từng bước

    with torch.no_grad():
        for i in range(1, nb_steps + 1):
            t_val = (i - 1) / nb_steps
            t = torch.full((1,), t_val, device=device, dtype=torch.float32)

            k_probs = bfn_model.discrete_output_distribution(theta, t)
            # Lưu top-5 max probs tại position 0
            top5, top5_idx = k_probs[0, 0].topk(min(5, bfn_model.K))
            theta_history.append({
                "step": i,
                "t": t_val,
                "max_prob": k_probs[0, 0].max().item(),
                "entropy": -(k_probs[0, 0] * (k_probs[0, 0] + eps_).log()).sum().item(),
                "top5_chars": [(itos[idx.item()], prob.item()) for idx, prob in zip(top5_idx, top5)],
            })

            # Update theta (simplified – không dùng stochastic sample để visualize clean)
            k = k_probs[0].argmax(dim=-1, keepdim=False)  # greedy
            e_k = F.one_hot(k, num_classes=bfn_model.K).float().unsqueeze(0)
            alpha = bfn_model.beta * (2 * i - 1) / (nb_steps ** 2)
            y = alpha * (bfn_model.K * e_k - 1)
            theta = F.softmax(y + torch.log(theta + eps_), dim=-1)

    return theta_history


print("📊 Tracking θ evolution during sampling (greedy mode)...")
history = visualize_theta_evolution(bfn, nb_steps=30, device=DEVICE)

# Plot entropy và max_prob
steps   = [h["step"] for h in history]
max_p   = [h["max_prob"] for h in history]
entropy = [h["entropy"] for h in history]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

ax1.plot(steps, max_p, 'o-', color='green', linewidth=2, markersize=4)
ax1.set_xlabel("Sampling Step", fontsize=12)
ax1.set_ylabel("Max Probability at Position 0", fontsize=12)
ax1.set_title("Confidence Grows with Bayesian Updates", fontsize=13)
ax1.axhline(1/VOCAB_SIZE, color='gray', linestyle='--', alpha=0.7, label=f'Random baseline (1/{VOCAB_SIZE}≈{1/VOCAB_SIZE:.3f})')
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.plot(steps, entropy, 's-', color='purple', linewidth=2, markersize=4)
ax2.set_xlabel("Sampling Step", fontsize=12)
ax2.set_ylabel("Entropy H(θ) at Position 0", fontsize=12)
ax2.set_title("Entropy Decreases (More Certain)", fontsize=13)
ax2.axhline(math.log(VOCAB_SIZE), color='gray', linestyle='--', alpha=0.7, 
             label=f'Max entropy (uniform)')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.suptitle("BFN: Bayesian Flow θ Evolution During Sampling", fontsize=14)
plt.tight_layout()
plt.savefig("bfn_theta_evolution.png", dpi=150, bbox_inches='tight')
plt.show()

print("\n🔝 Top-5 predictions at position 0 (final step):")
last = history[-1]
for char, prob in last["top5_chars"]:
    bar = "█" * int(prob * 40)
    print(f"  '{char}' [{repr(char)}]: {prob:.3f} {bar}")


---
## 10. Save & Load Model


In [ ]:
# ─────────────────────────────────────────────────────────────
# Save checkpoint
# ─────────────────────────────────────────────────────────────

checkpoint = {
    "model_state_dict": bfn.state_dict(),
    "optimizer_state_dict": optimizer.state_dict(),
    "config": {
        "vocab_size":  config.vocab_size,
        "max_seq_len": config.max_seq_len,
        "n_embd":      config.n_embd,
        "n_heads":     config.n_heads,
        "n_layers":    config.n_layers,
        "dropout":     config.dropout,
    },
    "step": step,
    "train_losses": train_losses,
    "vocab": {"stoi": stoi, "itos": itos},
}

CKPT_PATH = "bfn_gpt2_checkpoint.pt"
torch.save(checkpoint, CKPT_PATH)
print(f"✅ Model saved to {CKPT_PATH}")
print(f"   Checkpoint size: {__import__('os').path.getsize(CKPT_PATH) / 1024:.1f} KB")


In [ ]:
# ─────────────────────────────────────────────────────────────
# Load checkpoint (demo)
# ─────────────────────────────────────────────────────────────

def load_bfn_from_checkpoint(ckpt_path: str, device: str = "cpu"):
    """Load model từ checkpoint."""
    ckpt = torch.load(ckpt_path, map_location=device)
    cfg  = ckpt["config"]

    # Recreate config
    loaded_config = GPT2Config(
        vocab_size  = cfg["vocab_size"],
        max_seq_len = cfg["max_seq_len"],
        n_embd      = cfg["n_embd"],
        n_heads     = cfg["n_heads"],
        n_layers    = cfg["n_layers"],
        dropout     = cfg["dropout"],
    )

    # Recreate model
    loaded_backbone = GPT2BFN(loaded_config).to(device)
    loaded_bfn = BayesianFlowNetwork(
        model = loaded_backbone,
        D     = loaded_config.max_seq_len,
        K     = loaded_config.vocab_size,
        beta  = 1.0,
    ).to(device)

    loaded_bfn.load_state_dict(ckpt["model_state_dict"])
    loaded_bfn.eval()

    vocab = ckpt["vocab"]
    decode_fn = lambda ids: "".join([vocab["itos"][str(i)] for i in ids 
                                      if str(i) in vocab["itos"]])

    print(f"✅ Loaded checkpoint from step {ckpt['step']}")
    return loaded_bfn, decode_fn


# Test load
print("🔄 Testing checkpoint load...")
loaded_model, loaded_decode = load_bfn_from_checkpoint(CKPT_PATH, device=DEVICE)
print("   Model loaded successfully!")

# Quick sanity check
test_x = torch.randint(0, VOCAB_SIZE, (2, SEQ_LEN), device=DEVICE)
test_loss = loaded_model.process(test_x)
print(f"   Sanity check loss: {test_loss.item():.4f}")


---
## 11. Scaling lên GPT-2 Full Scale

Để reproduce "Bayesian Flow GPT-2 Scale" đầy đủ như trong mục TODO của repo:

### 11.1 Thay Dataset

```python
# Dùng GPT-2 tokenizer thay cho character-level
from transformers import GPT2Tokenizer
tokenizer = GPT2Tokenizer.from_pretrained("gpt2")

# Dùng WikiText-103 hoặc OpenWebText
dataset = load_dataset("wikitext", "wikitext-103-raw-v1")
```

### 11.2 Thay Config

```python
config = GPT2Config(
    vocab_size  = 50257,   # GPT-2 BPE vocab
    max_seq_len = 1024,    # GPT-2 context length
    n_embd      = 768,     # GPT-2 small
    n_heads     = 12,
    n_layers    = 12,
    dropout     = 0.1,
)
# Tổng: ~125M parameters (kể cả BFN input_proj)
```

### 11.3 Training Scale

| Config | Parameters | GPU | Est. Time |
|--------|-----------|-----|-----------|
| Demo (this notebook) | ~2M | CPU | 10 min |
| GPT-2 small | ~125M | 1x A100 | ~2 days |
| GPT-2 medium | ~350M | 4x A100 | ~1 week |

### 11.4 Key differences so với diffusion models

$$\underbrace{p_\theta(\mathbf{x})}_{\text{BFN}} = \int p_F(\boldsymbol{\theta} \mid \mathbf{x}; t) \, p_O(\mathbf{x} \mid \boldsymbol{\theta}, t) \, dt$$

Ưu điểm của BFN so với MDLM/D3PM cho text:
1. **Không cần masking** – flow trong parameter space, không phải input space
2. **Continuous time** – không discretize timesteps
3. **Flexible** – cùng framework hoạt động cho continuous, discrete, và ordinal data

---
### References

1. Graves, A., et al. (2023). *Bayesian Flow Networks*. arXiv:2308.07037
2. Radford, A., et al. (2019). *Language Models are Unsupervised Multitask Learners* (GPT-2)
3. Ho, J., et al. (2020). *Denoising Diffusion Probabilistic Models*
4. Austin, J., et al. (2021). *Structured Denoising Diffusion Models in Discrete State-Spaces* (D3PM)
